In [77]:
import pyexasol
import configparser
import pandas as pd
from haversine import haversine, Unit

pd.set_option('max_columns', None)

In [23]:
def conn(config_filepath, schema, env):
    """Exasol connection"""
    env=env.upper()
    config = configparser.ConfigParser()
    file = config_filepath+schema+env+'.ini'
    config.read(file)
    key=schema.lower()+env
    return pyexasol.connect(dsn=config[key]['dsn'], user=config[key]['user'], password=config[key]['pwd'], schema=config[key]['schema'])

In [24]:
config_filepath='C:/Users/USER/.spyder-py3/'
env='prod'
schema='pfx'
connect=conn(config_filepath,schema, env)
df = connect.export_to_pandas("SELECT * FROM DWHPFX.ERFP_POI_CLUSTER WHERE UPPER(RFP_PHASE) LIKE 'FINAL_BID_ANALYSIS'  AND RFP_ID='00000000-0000-0000-0000-000000000000'")
df.head()

In [64]:
lat_long = ['RFP_POI_LATITUDE','RFP_POI_LONGITUDE','RFP_HOTEL_LATITUDE', 'RFP_HOTEL_LONGITUDE']
for item in lat_long:
    df[item] = df[item].astype('float64')

df['DISTANCE_KMS'] = df.apply(lambda row: haversine((row['RFP_POI_LATITUDE'],row['RFP_POI_LONGITUDE']), \
                                        (row['RFP_HOTEL_LATITUDE'],row['RFP_HOTEL_LONGITUDE'])
                                       ),axis=1
                              )
df.head()                              

In [101]:
# tmp =[]
# df1=df.iloc[:,:-1]
# ## iterrows approach
# for i, row in df1.iterrows():
# #     print(i, row['RFP_POI_LATITUDE'])
#     hs=haversine((row['RFP_POI_LATITUDE'],row['RFP_POI_LONGITUDE']),(row['RFP_HOTEL_LATITUDE'],row['RFP_HOTEL_LONGITUDE']))
#     tmp.append(hs)

# df1['DISTANCE_KM']= tmp
# df1.head(10)

In [83]:
df1.columns

In [87]:
import scipy.cluster.hierarchy as shc
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(10, 7))
plt.title("Dendograms")
dend = shc.dendrogram(shc.linkage(df1[['DISTANCE_KM']], method='ward'))

In [91]:
from sklearn.cluster import AgglomerativeClustering

cluster = AgglomerativeClustering(n_clusters=5, affinity='euclidean', linkage='ward')
df1['CLUSTERS']=cluster.fit_predict(df1[['DISTANCE_KM']])

In [100]:
plt.figure(figsize=(10, 7))
plt.scatter(df1['RFP_HOTEL_ID'], df1['DISTANCE_KM'], c=cluster.labels_, cmap='rainbow')

In [93]:
df1.head()

In [99]:
cluster.labels_

In [104]:
df1[['CLUSTERS','DISTANCE_KM']].groupby('CLUSTERS').mean()

In [105]:
df1[df1['CLUSTERS']==2]

In [106]:
df1[df1['CLUSTERS']==3]

In [107]:
df1[['CLUSTERS','RFP_HOTEL_ID']].groupby('CLUSTERS').count()